# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** Prior attempt stopped during output capture: local DGX CLI images were mistaken for remote files, then external-directory access was auto-rejected. Separate cell logs exist, but a full executed notebook was not saved. A saved-run retry is in progress; this copy is an unverified reference snapshot.

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: Simultaneous and Dynamic Super-Resolution Imaging of Two Proteins in Arabidopsis thaliana using dual-color sptPALM

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-ba72ff372a5624ef5f676f99fee55860)


# Reproducing the diffusion-coefficient statistics of dual-color sptPALM (OneFlowTraX batch analysis)

**Paper:** Rohr L., Rausch L., Ehinger A., Burmeister N. G., Meixner A. J., Kemmerling B., Harter K., zur Oven-Krockhaus S. — *Simultaneous and Dynamic Super-Resolution Imaging of Two Proteins in *Arabidopsis thaliana* using dual-color sptPALM*, bioRxiv 2024.03.13.584811 (v2 retitled: "Establishing Dual-Color Super-Resolution sptPALM for Live-Cell Plasma Membrane Protein Tracking in *Arabidopsis thaliana*"). DOI: [10.1101/2024.03.13.584811](https://doi.org/10.1101/2024.03.13.584811)

**Author code:** [OneFlowTraX](https://github.com/svenzok/OneFlowTraX) (MATLAB App, GNU GPL), pinned commit `c1a5e937113bda39f64b3d7f6c0fa264348e57b5`.

---

## ⚠️ Scope of this notebook (partial demonstration)

OneFlowTraX is a **MATLAB App Designer application** (requires MATLAB R2026a + toolboxes and Windows-only `.mexw64` localization binaries); its full localization→tracking→MSD pipeline **cannot run in Colab**. Microscopy acquisition and the custom channel-separation script are likewise out of scope.

This notebook instead reproduces the paper's **quantitative diffusion statistics** from the authors' own deposited OneFlowTraX **batch-analysis results** (repository `Figure data/Figure 5/`, figure numbering of bioRxiv **v1**; v2 condensed the figures): per-cell log10(D) Gaussian peak fits and Kruskal–Wallis comparisons of diffusion coefficients of RLP44 and BRI1 fused to PA-GFP, PATagRFP, or mEos3.2 (raw fluorescence-tag comparison, Fig. 5C of v1).

> **AI-assisted statistics translation / AI による統計解析の移植について:** The authors' R statistics script is available "upon request" only and MATLAB cannot be executed here. The Gaussian peak-D computation below is an **exact port** of the authors' `batchAnalysis.m` (validated against the authors' own stored per-cell results inside this notebook). The Shapiro–Wilk / Kruskal–Wallis tests are an **AI reimplementation** following the paper's methods description (scipy.stats), not author code. The authors did not provide this Colab implementation. The executed example and listed checks were validated, but untested behavior is not guaranteed.

## Runtime

**No GPU is needed.** Select the plain **CPU** runtime in Colab (Runtime → Change runtime type → CPU). Everything runs in a few seconds.

## Environment check

In [ ]:
import sys, platform, time
print('python:', platform.python_version())
import numpy as np, pandas as pd, scipy, matplotlib
print('numpy:', np.__version__, '| pandas:', pd.__version__, '| scipy:', scipy.__version__, '| matplotlib:', matplotlib.__version__)
import importlib.util
print('openpyxl available:', importlib.util.find_spec('openpyxl') is not None)
print('CPU run started:', time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime()))

## 1. Download the authors' deposited Figure 5 batch results

Measured data files are downloaded **inside Colab** from the authors' pinned GitHub repository (GNU GPL; data downloaded transiently for this analysis). SHA-256 of every file is recorded for provenance.

In [ ]:
import hashlib, os, urllib.request

REPO = 'svenzok/OneFlowTraX'
COMMIT = 'c1a5e937113bda39f64b3d7f6c0fa264348e57b5'
BASE = f'https://raw.githubusercontent.com/{REPO}/{COMMIT}/'
FILES = [
    'Figure data/Figure 5/5C_BRI1/OFT_5C_BRI1_PA-GFP.xlsx',
    'Figure data/Figure 5/5C_BRI1/OFT_5C_BRI1_PATagRFP.xlsx',
    'Figure data/Figure 5/5C_BRI1/OFT_5C_BRI1_mEos3.2.xlsx',
    'Figure data/Figure 5/5C_RLP44/OFT_5C_RLP44_PA-GFP.xlsx',
    'Figure data/Figure 5/5C_RLP44/OFT_5C_RLP44_PATagRFP.xlsx',
    'Figure data/Figure 5/5C_RLP44/OFT_5C_RLP44_mEos3.2.xlsx',
    'Figure data/Figure 5/Fig5_Results.xlsx',
]
DATA_DIR = '/content/figure_data'
os.makedirs(DATA_DIR, exist_ok=True)

for rel in FILES:
    url = BASE + urllib.parse.quote(rel)
    dest = os.path.join(DATA_DIR, os.path.basename(rel))
    if not os.path.exists(dest):
        with urllib.request.urlopen(url, timeout=120) as r, open(dest, 'wb') as f:
            f.write(r.read())
    raw = open(dest, 'rb').read()
    assert raw[:2] == b'PK', f'{rel}: not an xlsx (zip) payload'
    print(f'{os.path.basename(rel):36s} {len(raw):>9,d} bytes  sha256={hashlib.sha256(raw).hexdigest()[:16]}…')

## 2. Load the per-cell OneFlowTraX batch results

`batchAnalysis.m` writes, for every input file (= one imaged cell), a row into the sheet **`D fit (1 pop)`** with columns `file ID`, `N` (number of fitted per-cell D values) and `D (µm²/s)` (peak D = 10^mu of a Normal fit to log10 D), and the raw per-cell D values into **`D lists`** (one column per file, values from row 3 down). We parse both sheets defensively (sheet names are located case-insensitively) and record the group (protein × fluorophore tag) from the file name.

In [ ]:
import re
import urllib.parse  # (used above)

def find_sheet(xls, *keywords):
    for name in xls.sheet_names:
        low = name.lower()
        if all(k.lower() in low for k in keywords):
            return name
    return None

def load_condition(path, protein, tag):
    xls = pd.ExcelFile(path)
    fit_sheet = find_sheet(xls, 'd fit', '1 pop')
    list_sheet = find_sheet(xls, 'd lists')
    assert fit_sheet and list_sheet, f'{path}: expected OneFlowTraX sheets not found; has {xls.sheet_names}'

    fit = pd.read_excel(xls, sheet_name=fit_sheet)
    fit.columns = [str(c).strip() for c in fit.columns]
    dcol = [c for c in fit.columns if c.startswith('D')][0]
    fit = fit.rename(columns={'file ID': 'file', 'N': 'N', dcol: 'D_stored'})
    fit['file'] = fit['file'].astype(str)
    fit = fit[pd.to_numeric(fit['D_stored'], errors='coerce').notna()].copy()
    fit['D_stored'] = fit['D_stored'].astype(float)

    # 'D lists': column 0 = 'file ID' header; each following column is one file's D values (header row 1, label row 2, data from row 3)
    lists_raw = pd.read_excel(xls, sheet_name=list_sheet, header=None)
    per_cell = {}
    for col in range(1, lists_raw.shape[1]):
        fid = str(lists_raw.iat[0, col])
        vals = pd.to_numeric(lists_raw.iloc[2:, col], errors='coerce').dropna().to_numpy(dtype=float)
        if len(vals):
            per_cell[fid] = vals
    return {'protein': protein, 'tag': tag, 'fit': fit, 'dlists': per_cell,
            'sheets': xls.sheet_names}

CONDITIONS = [
    ('OFT_5C_RLP44_PA-GFP.xlsx',    'RLP44', 'PA-GFP'),
    ('OFT_5C_RLP44_PATagRFP.xlsx',  'RLP44', 'PATagRFP'),
    ('OFT_5C_RLP44_mEos3.2.xlsx',   'RLP44', 'mEos3.2'),
    ('OFT_5C_BRI1_PA-GFP.xlsx',     'BRI1',  'PA-GFP'),
    ('OFT_5C_BRI1_PATagRFP.xlsx',   'BRI1',  'PATagRFP'),
    ('OFT_5C_BRI1_mEos3.2.xlsx',    'BRI1',  'mEos3.2'),
]
data = {}
for fname, protein, tag in CONDITIONS:
    data[(protein, tag)] = load_condition(os.path.join(DATA_DIR, fname), protein, tag)
    d = data[(protein, tag)]
    print(f'{protein:6s} {tag:9s}: {len(d["fit"]):>3d} cells, '
          f'{sum(len(v) for v in d["dlists"].values()):>5d} fitted D values, '
          f'sheets={d["sheets"]}')

## 3. Exact port of the OneFlowTraX peak-D computation (validation against the authors' stored results)

`batchAnalysis.m` (lines ≈313–326, pinned commit):

```matlab
Dlist  = indivMSDfits(isValid, 1);
logDlist = log10(Dlist);
pd = fitdist(logDlist, 'Normal', 'Options', statOptions);   % MaxFunEvals = 500
D1pop = 10^pd.mu;   % peak diffusion coefficient (µm²/s)
```

MATLAB's `fitdist(...,'Normal')` is a maximum-likelihood Normal fit — identical to `scipy.stats.norm.fit` on `log10(D)`. We recompute the peak D from the deposited raw per-cell `D lists` and compare with the authors' stored `D fit (1 pop)` values; agreement confirms the port is exact.

In [ ]:
from scipy.stats import norm

rows = []
for (protein, tag), d in data.items():
    stored = d['fit']['D_stored'].to_numpy()
    recomputed, agreement = [], []
    for _, r in d['fit'].iterrows():
        fid = r['file']
        vals = d['dlists'].get(fid)
        if vals is None or len(vals) == 0:
            continue
        vals = vals[np.isfinite(vals) & (vals > 0)]
        if len(vals) == 0:
            continue
        mu, sd = norm.fit(np.log10(vals))
        recomputed.append(10 ** mu)
        agreement.append(abs(10 ** mu - r['D_stored']) / max(r['D_stored'], 1e-30))
        d['dlists'][fid] = vals
    rows.append({'protein': protein, 'tag': tag, 'n_cells_stored': len(stored),
                 'n_cells_checked': len(recomputed),
                 'max_rel_diff': max(agreement) if agreement else float('nan')})

check = pd.DataFrame(rows)
print(check.to_string(index=False))
worst = check['max_rel_diff'].max()
print(f'\nWorst relative deviation recomputed vs authors-stored peak D: {worst:.3e}')
assert worst < 1e-6, 'peak-D port does not match the stored OneFlowTraX results'
print('PASS: recomputed peak D matches the authors stored D fit (1 pop) values (rtol < 1e-6).')

## 4. Per-cell log10(D) distributions and Gaussian peak fits (paper Fig. 5C-style)

Per-cell `log10(D)` distributions per condition with the fitted Normal curves; peak D values are the fitted means (µm²/s).

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=True)
colors = {'PA-GFP': '#2c7bb6', 'PATagRFP': '#d7191c', 'mEos3.2': '#fdae61'}
PAIRS = [(protein, tag) for _, protein, tag in CONDITIONS]
for ax, (protein, tag) in zip(axes.ravel(), PAIRS):
    d = data[(protein, tag)]
    allD = np.concatenate(list(d['dlists'].values()))
    logD = np.log10(allD)
    mu, sd = norm.fit(logD)
    xs = np.linspace(-4.5, 1.5, 400)
    ax.hist(logD, bins=40, density=True, color=colors[tag], alpha=0.45, label=f'per-cell log10(D), N={len(logD)}')
    ax.plot(xs, norm.pdf(xs, mu, sd), 'k-', lw=1.8,
            label=f'Normal fit: peak D = {10**mu:.3g} µm²/s')
    ax.set_title(f'{protein}–{tag}', fontsize=11)
    ax.set_xlabel('log10(D)  [log10(µm²/s)]'); ax.set_ylabel('density')
    ax.legend(fontsize=8)
fig.suptitle('Per-cell log10(D) distributions with OneFlowTraX Gaussian peak fits (deposited Figure 5 data)', y=1.02)
fig.tight_layout()
fig.savefig('/content/fig_logD_distributions.png', dpi=110, bbox_inches='tight')
plt.show()

Peak-D boxplots per condition (boxes: median, IQR, 1.5×IQR whiskers, as in the paper's figure style) on a log scale; diamonds mark the pooled Gaussian peak D per condition.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
order = [(p, t) for p in ('RLP44', 'BRI1') for t in ('PA-GFP', 'PATagRFP', 'mEos3.2')]
positions, boxdata, labels, cols = [], [], [], []
for i, key in enumerate(order):
    d = data[key]
    peak = d['fit']['D_stored'].to_numpy()
    positions.append(i); boxdata.append(peak)
    labels.append(f'{key[0]}\n{key[1]}'); cols.append(colors[key[1]])
bp = ax.boxplot(boxdata, positions=positions, widths=0.6, patch_artist=True,
                whiskerprops=dict(color='k'), medianprops=dict(color='k'))
for patch, c in zip(bp['boxes'], cols):
    patch.set_facecolor(c); patch.set_alpha(0.55)
for i, key in enumerate(order):
    d = data[key]
    allD = np.concatenate(list(d['dlists'].values()))
    mu, _ = norm.fit(np.log10(allD))
    ax.scatter([i], [10 ** mu], marker='D', color='k', zorder=5,
               label='Gaussian peak D (pooled log10 fit)' if i == 0 else None)
ax.set_xticks(positions); ax.set_xticklabels(labels)
ax.set_ylabel('peak D per cell (µm²/s)')
ax.set_yscale('log')
ax.set_title('Peak diffusion coefficients per cell (authors\' stored per-cell Gaussian fits)')
ax.legend()
fig.tight_layout()
fig.savefig('/content/fig_peakD_boxplots.png', dpi=110, bbox_inches='tight')
plt.show()

summary = []
for (protein, tag), d in data.items():
    allD = np.concatenate(list(d['dlists'].values()))
    mu, sd = norm.fit(np.log10(allD))
    summary.append({'protein': protein, 'tag': tag, 'n_cells': len(d['fit']),
                    'n_D_values': len(allD),
                    'peak_D_um2_s': 10 ** mu,
                    'median_D_um2_s': float(np.median(allD))})
summary_df = pd.DataFrame(summary)
summary_df

## 5. Statistics (Shapiro–Wilk + Kruskal–Wallis; AI-reimplemented R script)

Following the paper's statistics phase: Shapiro–Wilk normality tests on log10(D) per condition, followed by Kruskal–Wallis tests. The authors' R script (R 4.4.1, ggplot2/ggpubr) is only available "upon request"; these tests are an **AI reimplementation** with `scipy.stats`.

Tests performed:
1. **Within each protein:** Kruskal–Wallis across the three fluorophore tags (does diffusion depend on the tag?).
2. **Between proteins:** Kruskal–Wallis RLP44 vs BRI1 (per tag) — the paper's finding that BRI1 is less mobile than RLP44.

In [ ]:
from scipy.stats import shapiro, kruskal

print('Shapiro–Wilk normality tests on per-cell log10(D):')
sw_rows = []
for (protein, tag), d in data.items():
    allD = np.concatenate(list(d['dlists'].values()))
    stat, p = shapiro(np.log10(allD))
    sw_rows.append({'protein': protein, 'tag': tag, 'W': stat, 'p_value': p})
print(pd.DataFrame(sw_rows).to_string(index=False))

print('\nKruskal–Wallis across the three fluorophore tags, within each protein:')
kw_rows = []
for protein in ('RLP44', 'BRI1'):
    groups = [np.concatenate(list(data[(protein, t)]['dlists'].values())) for t in ('PA-GFP', 'PATagRFP', 'mEos3.2')]
    stat, p = kruskal(*[np.log10(g) for g in groups])
    kw_rows.append({'comparison': f'{protein}: tag effect', 'H': stat, 'p_value': p})
for t in ('PA-GFP', 'PATagRFP', 'mEos3.2'):
    g_r = np.concatenate(list(data[('RLP44', t)]['dlists'].values()))
    g_b = np.concatenate(list(data[('BRI1', t)]['dlists'].values()))
    stat, p = kruskal(np.log10(g_r), np.log10(g_b))
    kw_rows.append({'comparison': f'RLP44 vs BRI1 ({t})', 'H': stat, 'p_value': p})
kw_df = pd.DataFrame(kw_rows)
kw_df['significant_alpha_0.05'] = kw_df['p_value'] < 0.05
kw_df

## 6. Cross-check against the authors' aggregated `Fig5_Results.xlsx`

This file was produced by the authors' (unavailable) R plotting/statistics script; we parse it defensively and display its sheets as historical reference values for comparison with our recomputation.

In [ ]:
res_path = os.path.join(DATA_DIR, 'Fig5_Results.xlsx')
xls = pd.ExcelFile(res_path)
print('sheets:', xls.sheet_names)
for name in xls.sheet_names[:8]:
    df = pd.read_excel(xls, sheet_name=name, header=None, nrows=12)
    print(f'--- sheet: {name!r}  shape(first 12 rows): {pd.read_excel(xls, sheet_name=name, header=None).shape}')
    print(df.to_string(max_cols=8))
    print()

## Summary and limitations

**Reproduced here (executed, CPU):** from the authors' deposited OneFlowTraX batch-analysis results for the fluorescence-tag comparison (repository `Figure data/Figure 5`, bioRxiv v1 Fig. 5C data), this notebook (a) downloaded the six per-condition xlsx files + the authors' aggregated results with SHA-256 provenance, (b) **exactly reproduced** the authors' peak-D computation (Gaussian/Normal MLE fit of per-cell log10(D); peak D = 10^mu) with agreement < 1e-6 relative deviation against the authors' stored per-cell results, (c) recomputed the paper's Shapiro–Wilk / Kruskal–Wallis statistics (AI reimplementation of the unavailable R script), and (d) rendered the per-cell log10(D) distributions, peak-D boxplots, and summary table.

**Out of scope (recorded, not hidden):** microscope acquisition, the Windows-only MATLAB localization/tracking/MSD pipeline (`.mexw64` MLE fitters), the custom channel-separation MATLAB script, and the raw dual-color movies (not publicly deposited; sample stacks only via the authors' Google Drive). The repository's deposited `Figure data` files carry no explicit license beyond the repository's GNU GPL for scripts; they are downloaded transiently for analysis here.

**Provenance:** OneFlowTraX pinned commit `c1a5e937113bda39f64b3d7f6c0fa264348e57b5` (retrieved 2026-10-01); paper bioRxiv DOI 10.1101/2024.03.13.584811. Cite OneFlowTraX as: Rohr et al., *Front. Plant Sci.* 15:1358935 (2024), doi:10.3389/fpls.2024.1358935.